# Phase 4C.1: Colab T4 Training Launcher (Persistent Drive Architecture)

- **Lưu trữ**: Google Drive lưu trữ bền vững cả input artifacts và output results.
- **Workspace**: `/content` làm không gian staging và training workspace cho Colab.
- **Phần cứng**: NVIDIA T4 GPU trên Colab runtime.
- **Chuẩn bị**: Đặt đủ 3 artifacts vào thư mục Drive `inputs/`:
  1. `phase_4c1_code_79bb115.tar.gz`
  2. `phase_4c1_binary_n250_reusable.tar`
  3. `phase_4c1_t4_execute_all_stage1.sh`
- **Thực thi**: Đổi `EXECUTE = True` tại Cell 1 và chọn **Runtime > Run all**.
- **An toàn**: Không chạy đồng thời operator script trong Terminal khi notebook đang chạy.

In [ ]:
# Cell 1: Configuration
from pathlib import Path

# Execution switches (Default: False for safety)
EXECUTE = False
DOWNLOAD_FINAL_ARCHIVE = False

# Google Drive storage paths
DRIVE_MOUNT = Path("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/forensics-web-lab/phase_4c1")
DRIVE_INPUT_DIR = DRIVE_ROOT / "inputs"
DRIVE_OUTPUT_DIR = DRIVE_ROOT / "runs" / "execution_79bb115"

# Local Colab workspace paths
LOCAL_TRANSFER_DIR = Path("/content/forensics-transfer")
LOCAL_OUTPUT_PATH = Path("/content/phase_4c1_outputs")

# Sealed immutable artifacts specification (Commit: 79bb115)
EXPECTED_ARTIFACTS = {
    "code_archive": {
        "filename": "phase_4c1_code_79bb115.tar.gz",
        "bytes": 758350,
        "sha256": "5e775a7708d1555bf22f56dceb5358e26e07dd224c4b6186f575aff4d2b590d5"
    },
    "reusable_bundle": {
        "filename": "phase_4c1_binary_n250_reusable.tar",
        "bytes": 724633600,
        "sha256": "d49a106f0c4991ca8d79776277cbf7331df209157725c438288720dc42226a27"
    },
    "operator_script": {
        "filename": "phase_4c1_t4_execute_all_stage1.sh",
        "bytes": 20005,
        "sha256": "deb3f04dd7c1039c4f3248f98a27f4eaf8e11f206a218125e949e5869da8ce05"
    }
}

print("Configuration loaded.")
print(f"  EXECUTE = {EXECUTE}")
print(f"  DOWNLOAD_FINAL_ARCHIVE = {DOWNLOAD_FINAL_ARCHIVE}")
print(f"  Drive Inputs:  {DRIVE_INPUT_DIR}")
print(f"  Drive Outputs: {DRIVE_OUTPUT_DIR}")


In [ ]:
# Cell 2: Mount Drive and Preflight
import sys
import os
import shutil
import hashlib
import torch

# 1. Mount Google Drive once
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    print("[WARN] google.colab not available; skipping live drive mount (non-Colab environment).")

# 2. Check platform and runtime
print(f"Platform: {sys.platform}")
print(f"Python:   {sys.version.split()[0]}")
print(f"PyTorch:  {torch.__version__}")
print(f"CUDA:     {torch.cuda.is_available()}")

gpu_model = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
print(f"GPU:      {gpu_model}")

assert sys.platform.startswith("linux"), f"Linux required, got {sys.platform}"

if EXECUTE:
    assert torch.cuda.is_available(), "CUDA must be available when EXECUTE=True"
    assert "T4" in gpu_model.upper(), f"NVIDIA T4 required, got {gpu_model}"

# 3. Ensure Drive directories exist
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_INPUT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 4. Check disk space on /content (require at least 5 GB free)
stat_content = shutil.disk_usage("/content") if Path("/content").exists() else shutil.disk_usage(".")
free_gb = stat_content.free / (1024 ** 3)
print(f"Available disk space: {free_gb:.2f} GB")
if EXECUTE:
    assert free_gb >= 5.0, f"Insufficient disk space on /content: {free_gb:.2f} GB available, need >= 5.0 GB"

# 5. Helper for streaming SHA-256 (1 MiB chunks)
def compute_sha256_streaming(file_path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(file_path, "rb") as f:
        while chunk := f.read(chunk_size):
            h.update(chunk)
    return h.hexdigest().lower()

# 6. Verify artifacts in Drive input directory
missing_artifacts = []
for key, spec in EXPECTED_ARTIFACTS.items():
    fp = DRIVE_INPUT_DIR / spec["filename"]
    if not fp.exists():
        missing_artifacts.append(spec["filename"])
        continue
    act_bytes = fp.stat().st_size
    assert act_bytes == spec["bytes"], (
        f"Byte mismatch in Drive for {spec['filename']}: expected {spec['bytes']}, got {act_bytes}"
    )
    act_sha = compute_sha256_streaming(fp)
    assert act_sha == spec["sha256"].lower(), (
        f"SHA-256 mismatch in Drive for {spec['filename']}: expected {spec['sha256']}, got {act_sha}"
    )
    print(f"[+] Verified in Drive: {spec['filename']} ({act_bytes:,} bytes, SHA-256 match)")

if missing_artifacts:
    msg = (
        f"Missing artifact(s) in Drive input directory: {missing_artifacts}\n"
        f"Please upload required files to: {DRIVE_INPUT_DIR}"
    )
    raise FileNotFoundError(msg)

print("[+] Preflight verification completed successfully.")


In [ ]:
# Cell 3: Staging and Persistent Output Binding
import os
import shutil
from pathlib import Path

LOCAL_TRANSFER_DIR.mkdir(parents=True, exist_ok=True)

# 1. Stage artifacts from Drive to /content/forensics-transfer using atomic .part copy
for key, spec in EXPECTED_ARTIFACTS.items():
    src = DRIVE_INPUT_DIR / spec["filename"]
    dst = LOCAL_TRANSFER_DIR / spec["filename"]
    need_copy = True
    if dst.exists():
        if dst.stat().st_size == spec["bytes"] and compute_sha256_streaming(dst) == spec["sha256"].lower():
            print(f"[+] Reusing verified local copy: {dst}")
            need_copy = False
        else:
            raise ValueError(f"Existing local file {dst} has incorrect bytes or SHA-256. Failing closed.")
    if need_copy:
        part_dst = LOCAL_TRANSFER_DIR / f"{spec['filename']}.part"
        if part_dst.exists():
            part_dst.unlink()
        print(f"[*] Copying {src.name} from Drive to local staging...")
        shutil.copy2(src, part_dst)
        part_bytes = part_dst.stat().st_size
        assert part_bytes == spec["bytes"], f"Byte mismatch on copy for {part_dst.name}"
        part_sha = compute_sha256_streaming(part_dst)
        assert part_sha == spec["sha256"].lower(), f"SHA-256 mismatch on copy for {part_dst.name}"
        part_dst.rename(dst)
        print(f"[+] Staged successfully: {dst}")

# Set execute permission on operator script
operator_local = LOCAL_TRANSFER_DIR / "phase_4c1_t4_execute_all_stage1.sh"
operator_local.chmod(0o755)

# 2. Bind persistent output directory: /content/phase_4c1_outputs -> DRIVE_OUTPUT_DIR
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if LOCAL_OUTPUT_PATH.exists() or LOCAL_OUTPUT_PATH.is_symlink():
    if LOCAL_OUTPUT_PATH.is_symlink():
        target_resolved = os.path.realpath(LOCAL_OUTPUT_PATH)
        expected_resolved = os.path.realpath(DRIVE_OUTPUT_DIR)
        if target_resolved != expected_resolved:
            raise RuntimeError(
                f"Symlink mismatch: {LOCAL_OUTPUT_PATH} points to {target_resolved}, expected {expected_resolved}."
            )
        print(f"[+] Output symlink verified: {LOCAL_OUTPUT_PATH} -> {DRIVE_OUTPUT_DIR}")
    else:
        raise RuntimeError(
            f"{LOCAL_OUTPUT_PATH} exists as a physical directory rather than a symlink to Drive. "
            "Failing closed to prevent overwriting or data loss."
        )
else:
    LOCAL_OUTPUT_PATH.symlink_to(DRIVE_OUTPUT_DIR, target_is_directory=True)
    print(f"[+] Created persistent output symlink: {LOCAL_OUTPUT_PATH} -> {DRIVE_OUTPUT_DIR}")

print("[+] Staging and persistent output binding complete.")


In [ ]:
# Cell 4: Execution
import subprocess

if not EXECUTE:
    print("[DRY-RUN] EXECUTE is False. Operator not invoked.")
    print("[DRY-RUN] Set EXECUTE = True in Cell 1 and run again to execute the 15-run Stage 1 matrix.")
else:
    operator_cmd = ["bash", str(LOCAL_TRANSFER_DIR / "phase_4c1_t4_execute_all_stage1.sh")]
    print(f"[*] Launching operator: {' '.join(operator_cmd)}")
    try:
        subprocess.run(operator_cmd, check=True)
        print("[+] Operator finished successfully.")
    except subprocess.CalledProcessError as e:
        print(f"[-] Operator exited with error code {e.returncode}.")
        print(f"[-] All completed runs and partial outputs in Drive ({DRIVE_OUTPUT_DIR}) are preserved intact.")
        raise


In [ ]:
# Cell 5: Audit and Result Handoff
import json
import hashlib
from pathlib import Path

REQUIRED_ARCHIVES = [
    "n50_results.tar.gz",
    "n100_results.tar.gz",
    "n250_results.tar.gz",
    "phase_4c1_t4_execution_logs.tar.gz",
    "phase_4c1_all_15_runs_results.tar.gz"
]

if not EXECUTE:
    print("[DRY-RUN] EXECUTE is False. Audit and handoff skipped.")
else:
    drive_download_dir = DRIVE_OUTPUT_DIR / "download"
    assert drive_download_dir.exists(), f"Download directory missing in Drive: {drive_download_dir}"
    print("Auditing persistent archives in Drive:")
    for name in REQUIRED_ARCHIVES:
        archive_path = drive_download_dir / name
        sidecar_path = drive_download_dir / f"{name}.sha256"
        assert archive_path.exists(), f"Archive missing: {archive_path}"
        assert sidecar_path.exists(), f"Sidecar checksum missing: {sidecar_path}"
        bytes_count = archive_path.stat().st_size
        assert bytes_count > 0, f"Archive is empty: {archive_path}"
        sidecar_content = sidecar_path.read_text(encoding="utf-8").strip().split()
        expected_hash = sidecar_content[0].lower()
        actual_hash = compute_sha256_streaming(archive_path)
        assert actual_hash == expected_hash, (
            f"Checksum mismatch for {name}: sidecar says {expected_hash}, calculated {actual_hash}"
        )
        print(f"  [PASS] {name} ({bytes_count:,} bytes) - SHA-256 match")
    completed_runs = 0
    for size in (50, 100, 250):
        for seed in (42, 1337, 2025, 3407, 9001):
            run_dir = DRIVE_OUTPUT_DIR / f"n{size}_seed_{seed}"
            receipt_path = run_dir / "run_receipt.json"
            assert receipt_path.exists(), f"Missing receipt for run: {run_dir.name}"
            with open(receipt_path, "r", encoding="utf-8") as f:
                receipt = json.load(f)
            assert receipt.get("status") == "completed", f"Run {run_dir.name} status not completed: {receipt.get('status')}"
            assert receipt.get("sample_size") == size, f"Sample size mismatch: {run_dir.name}"
            assert receipt.get("seed") == seed, f"Seed mismatch: {run_dir.name}"
            completed_runs += 1
    assert completed_runs == 15, f"Expected 15 completed runs, got {completed_runs}"
    print(f"[+] Verified all {completed_runs} runs completed successfully in Drive.")
    print(f"[+] Output directory in Drive: {DRIVE_OUTPUT_DIR}")
    if DOWNLOAD_FINAL_ARCHIVE:
        final_tar = drive_download_dir / "phase_4c1_all_15_runs_results.tar.gz"
        print(f"[*] Downloading {final_tar.name} to local browser...")
        try:
            from google.colab import files
            files.download(str(final_tar))
        except ImportError:
            print("[WARN] google.colab.files not available; skipping browser download.")
